Code to test Vector_Search_RAG.py

In [1]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [2]:
import sys
import os

# Add the parent folder "consigliami" to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Now import the class from scripts/DB_querying.py
from scripts.Vector_Search_RAG import SemanticReviewSearcher
from scripts.DB_querying import RestaurantRecommender

In [3]:
from pprint import pprint
from pymongo import MongoClient

# ========== Setup ==========

# Connect to MongoDB
client = MongoClient("localhost", 27017)
db = client["Reviews"]
collection = db["Barcelona"]

# Simulated User Query
user_query = "I want a place with great pizza and vegetarian options in a cozy environment"

# Step 1: Retrieve relevant reviews from MongoDB (e.g., matching 'pizza' or 'vegetarian')
keywords = ["pizza", "vegetarian", "cozy"]

mongo_filter = {
    "$or": [
        {"aspect_keys": {"$elemMatch": {"$regex": k, "$options": "i"}}}
        for k in keywords
    ]
}

# Retrieve reviews from MongoDB matching the keywords
retrieved_reviews = list(collection.find(mongo_filter).limit(10))  # limit for testing
print(f"📦 Retrieved {len(retrieved_reviews)} reviews from MongoDB.")


recomender = RestaurantRecommender(db_name = "Reviews", collection_name= "Barcelona", model_name= "llama3:8b")

#Selecting only the top resturant based on the numbers of reviews found
top_restaurants = recomender.enrich_results(retrieved_reviews)

context_reviews, top_resturants_list = recomender.fetch_full_reviews(top_restaurants=top_restaurants)


📦 Retrieved 6 reviews from MongoDB.
✅ MongoDB connected.
🏆 Top resturnats found: ['Sports_Bar_Italian_Food_C_Ample', 'Brunch_Bakery', 'We_love_italy_Pasta_Pizza_Restaurante_Barcelona']
Retrived 22 reviews from the most relevant restaurants


In [5]:

# ========== Step 2: Initialize and Build Temporary FAISS Index ==========
searcher = SemanticReviewSearcher()

searcher.build_index(context_reviews, content_field="review_full")

# ========== Step 3: Perform RAG ==========
response = searcher.run_rag(top_resturants_list, context_reviews, query=user_query)

# ========== Output ==========
print("\n💬 Final Answer from LLM:")
print(response)


c:\Users\jacop\Desktop\Lavori\Consigl_IA_mi-\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔧 Building FAISS index with 22 reviews...
✅ In-memory FAISS index built with 22 documents.

💬 Final Answer from LLM:
Based on the provided reviews, I highly recommend **Sports_Bar_Italian_Food_C_Ample** as the top choice for your query. Here's why:

1. **Great Pizza**: Several reviews praise the restaurant's pizzas, calling them "sublime," "amazing," and "the best in town." Even a review that mentions some disappointment with the pizza notes that it was still enjoyable.
2. **Vegetarian Options**: The restaurant has a variety of vegetarian options, including lasagna, vegetable pasta, and croqettas. Reviewers have mentioned enjoying these dishes, and one even specifies that the service is "vegetarian friendly."
3. **Cozy Environment**: While some reviewers mention that the atmosphere can be loud and bustling, others describe it as cozy and charming. The outdoor seating area with nice views (mentioned in a few reviews) might also provide a pleasant ambiance.

As for alternatives:

* **Bru